# TP 5 Geoloc 

Eléonore Veyron

## Echauffement : construire une URL

In [ ]:
def url(num,typ,nom):
    return f"https://api-adresse.data.gouv.fr/search/?q={num}+{typ}+{nom},Paris&limit=1"
num = 3
typ = "rue"
nom = "tourelles"
print(url(num,typ,nom))

https://api-adresse.data.gouv.fr/search/?q=3+rue+tourelles,Paris&limit=1


## Echauffement : sélectionner des lignes avec iloc et loc


In [ ]:

import pandas as pd

df = pd.DataFrame({
    "number": [3, 18, 7, 12, 5],
    "name": ["tourelles", "BERNARDINS", "Bouvart", "Rivoli", "Seine"],
})
sub = df[df.number > 10]

#1. sub devient df = pd.DataFrame({"number":[18,12],"name":["BERNARDINS","Rivoli"]})
# son index devient 1,3

#2. df.iloc[:2] = ({"number":[3,18],"name":["tourelles","BERNARDIN"]})
# df.loc[:2] = ({"number":[3,18,7],"name":["tourelles","BERNARDIN","Bouvart"]})
print(df.iloc[:2])
print(df.loc[:2])

#3. sub.iloc[:2] = ({"number":[18,12],"name":["BERNARDINS","Rivoli"]})
#sub.loc[:2] = ({"number":[18],"name":["BERNARDINS"]})

#4. sub.iloc[0] = 18, "BERNARDINS"
#sub.loc[0] = eror car pas d'index 0
print(sub.iloc[0])

#5. sub["name"].iloc[0] #on veut la 1ere ligne
#sub["name"].loc[1] #on regarde l'index
print(sub["name"].iloc[0])
print(sub["name"].loc[1]) # ou sub["name"].loc[sub.index[0]]


   number        name
0       3   tourelles
1      18  BERNARDINS
   number        name
0       3   tourelles
1      18  BERNARDINS
2       7     Bouvart
number            18
name      BERNARDINS
Name: 1, dtype: object
BERNARDINS
BERNARDINS


## Géolocalisation


In [ ]:
import pandas as pd

addresses = pd.read_csv('ARTEFACTS-addresses-f6f90e822bcd81e4b53409289753e9ba/data/addresses.csv')
addresses.iloc[:21].to_csv("addresses-small.csv",index=False) #sauvegarde le fichier
# OU : addresses.head(20).to_csv("....", index=False) car on veut pas d'index
addresses.head(4) 



,number,type,name,city
0,5,rue,RIQUET,Paris
1,141,boulevard,ITALIE,Paris
2,234,rue,FAUBOURG ST-HONORÉ,Paris
3,62,rue,MICHEL BIZOT,Paris


## Une par une

a) on obtient un dictionnaire, c'est toute les rues tourelles de Paris

b) avec &limit=1 on obtient seulement les informations pour la 1ere rue de tourelles

c) c'est le 1er sous-dico


In [ ]:

# requests is the swiss knife for doing http

import requests

def localize_one(num,typ,nom):
    url1 = url(num,typ,nom)
    r = requests.get(url1)
    if 200 <= r.status_code < 300: 
        return r.json()
    else : 
        return None 

import time

t1= time.time()
print(localize_one(18,'rue','BERNARDINS'))
t2= time.time()

print("Faire 30 000 requêtes prendrait", (t2-t1)*30000/3600, "h")
#c'est long car on importe tout le sous-dico

{'type': 'FeatureCollection', 'features': [{'type': 'Feature', 'geometry': {'type': 'Point', 'coordinates': [2.350728, 48.850345]}, 'properties': {'label': '18 Rue des Bernardins 75005 Paris', 'score': 0.7544495104895104, 'housenumber': '18', 'id': '75105_0896_00018', 'banId': 'acbebc10-eac0-475f-9466-e7dbe9446b02', 'name': '18 Rue des Bernardins', 'postcode': '75005', 'citycode': '75105', 'x': 652355.3, 'y': 6861340.65, 'city': 'Paris', 'district': 'Paris 5e Arrondissement', 'context': '75, Paris, Île-de-France', 'type': 'housenumber', 'importance': 0.68356, 'depcode': '75', 'street': 'Rue des Bernardins', '_type': 'address'}}], 'query': '18 rue BERNARDINS,Paris'}
Faire 30 000 requêtes prendrait 3.67694099744161 h


## En un seul coup - en python

In [ ]:


import requests

import io

def mass_post(filename, col_number, col_type, col_name, col_city):

    with open(filename) as feed:
        response = requests.post(
            "https://api-adresse.data.gouv.fr/search/csv/", 
            files={'data': feed},
            data={'columns': [col_number, col_type, col_name, col_city]})
    if not (200 <= response.status_code < 300):
        print(f"OOPS, got {response.status_code}")
        return None

    result = pd.read_csv(io.StringIO(response.text)) #car pd.read_csvp prend un dataframe

    return result


# votre code

def localize_many(filename, col_number, col_type, col_name, col_city):
    df = pd.read_csv(filename, encoding='utf-8')
    geoloc = mass_post(filename, col_number, col_type, col_name, col_city)
    geoloc = geoloc[['latitude', 'longitude', 'result_city', 'result_type']]
    return pd.merge(df,geoloc, how='inner', on=df.index)

# try your code on the small sample for starters

addresses_small = localize_many("addresses-small.csv", "number", "type", "name", "city")
addresses_small1 = addresses_small[addresses_small['result_city']=='Paris']
addresses_small1


,key_0,number,type,name,city,latitude,longitude,result_city,result_type
0,0,5,rue,RIQUET,Paris,48.888134,2.376448,Paris,housenumber
2,2,234,rue,FAUBOURG ST-HONORÉ,Paris,48.876160,2.302766,Paris,housenumber
3,3,62,rue,MICHEL BIZOT,Paris,48.835423,2.402040,Paris,housenumber
4,4,6,rue,OURS,Paris,48.863311,2.351775,Paris,housenumber
5,5,30,rue,PETIT-MUSC,Paris,48.852945,2.364722,Paris,housenumber
6,6,3,impasse,MASSONET,Paris,48.895498,2.351580,Paris,housenumber
7,7,56,rue,PANOYAUX,Paris,48.867023,2.388347,Paris,housenumber
8,8,16,rue,THÉRÈSE,Paris,48.866374,2.335212,Paris,housenumber
9,9,11,rue,JEAN ROBERT,Paris,48.889520,2.358170,Paris,housenumber
10,10,22,rue,FAUBOURG POISSONNIÈRE,Paris,48.872540,2.348002,Paris,housenumber


### Traitement par lots et le cache

In [ ]:

import os 

if os.path.exists("localize_many.csv"):
    t1 = time.time()
    addresse2 = pd.read_csv("localize_many.csv")
    t2=time.time()
    print("Temps de l'opération",(t2-t1)/60,"min")
    
else: 
    t1=time.time()

    batch_size=1000

    initial = pd.read_csv("ARTEFACTS-addresses-f6f90e822bcd81e4b53409289753e9ba/data/addresses.csv", encoding='utf-8')
    a=initial.iloc[0:1000].to_csv("loc", index=False)
    addresse2= localize_many("loc", "number", "type", "name", "city")

    for i in range(1,20):
        a=initial.iloc[i:i+batch_size].to_csv("loc", index=False)
        addresse3 = localize_many("loc", "number", "type", "name", "city")
        addresse2= pd.concat([addresse2,addresse3], ignore_index=True)
        addresse2.to_csv("localize_many.csv",index=False) #sauvegarde le fichier

    t2=time.time()
    print("Temps de l'opération",(t2-t1)/60,"min")



addresse2

#os.remove pour supprimer les fichiers temporaires des lots

Temps de l'opération 1.706658621629079 min
oui


,key_0,number,type,name,city,latitude,longitude,result_city,result_type
0,0,5,rue,RIQUET,Paris,48.888134,2.376448,Paris,housenumber
1,1,141,boulevard,ITALIE,Paris,46.669103,-1.418872,La Roche-sur-Yon,street
2,2,234,rue,FAUBOURG ST-HONORÉ,Paris,48.876160,2.302766,Paris,housenumber
3,3,62,rue,MICHEL BIZOT,Paris,48.835423,2.402040,Paris,housenumber
4,4,6,rue,OURS,Paris,48.863311,2.351775,Paris,housenumber
...,...,...,...,...,...,...,...,...,...
19995,995,229,boulevard,RASPAIL,Paris,48.839897,2.330662,Paris,housenumber
19996,996,3,rue,BAULANT,Paris,48.840910,2.386549,Paris,housenumber
19997,997,5,rue,GUYOT,Paris,48.891447,2.319708,Paris,housenumber
19998,998,23,rue,FAUBOURG SAINT-ANTOINE,Paris,48.852766,2.371394,Paris,housenumber


## Afficher sur une carte

In [ ]:


import folium 

# ~ chatelet
CENTER = 48.856542, 2.347614

## Le fond de carte

In [ ]:

# pour commencer on va recharger la dataframe précédente
import pandas as pd

addresses = pd.read_csv("ARTEFACTS-addresses-f6f90e822bcd81e4b53409289753e9ba/data/addresses-geoloc.csv")

# et en faire un petit échantillon
addresses_small = addresses.iloc[:20]

#Map centrée sur ce point et de zoom 13
def paris_map():
    return folium.Map(location= CENTER, zoom_start=13)
    
paris_map()

## On ajoute les adresses

In [38]:

addresses_small2 = addresses.iloc[:20]
addresses_small2.to_csv("adresses_small2.csv", index=False)


def map_addresses(geoloc):
    """
    creates folium map centered on Paris, with the various addresses
    contained in the input dataframe shown as a marker
    """
    m = paris_map()
    addresses_small2=geoloc.iloc[:20]
    location1=localize_many("adresses_small2.csv", "number", "type", "name", "city")
    for _, row in location1.iterrows():
        folium.Marker(
            location=(row["latitude_x"], row["longitude_x"]),
            tooltip="Click me!",
            icon=folium.Icon(icon="cloud"),
        ).add_to(m)
    return m

map_addresses(addresses)


## On sauve la carte 

In [ ]:

map_small = map_addresses(addresses_small)
map_small.save("addresses-small.html")


## Les quartiers de Paris 